<a href="https://colab.research.google.com/github/LINWOO0099/machine-learning/blob/main/Advanced_Modeling_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.impute import SimpleImputer # Added import
from sklearn.pipeline import Pipeline # Added import
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 1. Load data and define targets
# ------------------------------------------------------------
df = pd.read_csv('cleaned_data.csv')
print(df.head())
TARGET_REG = 'Price'          # <-- change to your column
y_reg = df[TARGET_REG]
y_clf = (y_reg > y_reg.median()).astype(int)
X = df.drop(columns=[TARGET_REG])
del df   # free original DataFrame

# ------------------------------------------------------------
# 2. Train/test split
# ------------------------------------------------------------
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)
del X, y_reg, y_clf   # free full data


  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS      12.0   
1  489434    79323P                   PINK CHERRY LIGHTS      12.0   
2  489434    79323W                  WHITE CHERRY LIGHTS      12.0   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE       48.0   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX      24.0   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


In [ ]:

# ------------------------------------------------------------
# 2. Build a preprocessing pipeline (for raw data)
# ------------------------------------------------------------
# Identify categorical and numeric columns
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Ordinal mappings (if any – modify as needed)
ordinal_mappings = {
    # 'education': ['High School', 'Bachelor', 'Master', 'PhD'],
}

# Build column transformer
transformers = []
for col in cat_cols:
    if col in ordinal_mappings:
        transformers.append(
            (col, OrdinalEncoder(categories=[ordinal_mappings[col]],
                                 handle_unknown='use_encoded_value',
                                 unknown_value=-1), [col])
        )
    else:
        transformers.append(
            (col, OneHotEncoder(drop='first', handle_unknown='ignore',
                                sparse_output=False,   # dense for compatibility
                                min_frequency=0.01, max_categories=20), [col])
        )

preprocessor = ColumnTransformer(transformers, remainder='passthrough')

# Full pipeline: preprocessing + scaling + classifier placeholder
# We'll create a base pipeline without a classifier and then clone it for each model.
base_preprocessor = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler())   # scaling is fine for tree models too
])

# We'll define a function to get a pipeline with a given classifier
def make_full_pipeline(classifier):
    return Pipeline([
        ('preproc', base_preprocessor),
        ('clf', classifier)
    ])

print("Categorical columns:", cat_cols)
print("Numerical columns:", num_cols)


Categorical columns: ['Invoice', 'StockCode', 'Description', 'InvoiceDate', 'Country']
Numerical columns: ['Quantity', 'Customer ID']


In [ ]:
# ------------------------------------------------------------
# 3. Decision Tree Baseline (unconstrained)
# ------------------------------------------------------------
dt_default = make_full_pipeline(DecisionTreeClassifier(random_state=42))
dt_default.fit(X_train, y_clf_train)
train_acc_def = accuracy_score(y_clf_train, dt_default.predict(X_train))
test_acc_def = accuracy_score(y_clf_test, dt_default.predict(X_test))
print("=== Decision Tree (default) ===")
print(f"Train accuracy: {train_acc_def:.4f}")
print(f"Test accuracy : {test_acc_def:.4f}\n")

=== Decision Tree (default) ===
Train accuracy: 0.7911
Test accuracy : 0.7609



In [ ]:
# ------------------------------------------------------------
# 4. Controlled Decision Tree (max_depth=5, min_samples_split=20)
# ------------------------------------------------------------
dt_controlled = make_full_pipeline(DecisionTreeClassifier(
    max_depth=5, min_samples_split=20, random_state=42
))
dt_controlled.fit(X_train, y_clf_train)
train_acc_ctrl = accuracy_score(y_clf_train, dt_controlled.predict(X_train))
test_acc_ctrl = accuracy_score(y_clf_test, dt_controlled.predict(X_test))
print("=== Decision Tree (controlled) ===")
print(f"Train accuracy: {train_acc_ctrl:.4f}")
print(f"Test accuracy : {test_acc_ctrl:.4f}\n")


=== Decision Tree (controlled) ===
Train accuracy: 0.6890
Test accuracy : 0.6873



In [ ]:
#------------------------------------------------------------
# 5. Gini vs Entropy (controlled, max_depth=5)
# ------------------------------------------------------------
dt_gini = make_full_pipeline(DecisionTreeClassifier(
    criterion='gini', max_depth=5, random_state=42
))
dt_gini.fit(X_train, y_clf_train)
test_acc_gini = accuracy_score(y_clf_test, dt_gini.predict(X_test))

dt_entropy = make_full_pipeline(DecisionTreeClassifier(
    criterion='entropy', max_depth=5, random_state=42
))
dt_entropy.fit(X_train, y_clf_train)
test_acc_entropy = accuracy_score(y_clf_test, dt_entropy.predict(X_test))

print("=== Gini vs Entropy (test accuracy) ===")
print(f"Gini    : {test_acc_gini:.4f}")
print(f"Entropy : {test_acc_entropy:.4f}\n")


=== Gini vs Entropy (test accuracy) ===
Gini    : 0.6873
Entropy : 0.6872



In [ ]:
# ------------------------------------------------------------
# 6. Random Forest (n_estimators=100, max_depth=10)
# ------------------------------------------------------------
rf = make_full_pipeline(RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42
))
rf.fit(X_train, y_clf_train)
rf_train_acc = accuracy_score(y_clf_train, rf.predict(X_train))
rf_test_acc = accuracy_score(y_clf_test, rf.predict(X_test))
rf_auc = roc_auc_score(y_clf_test, rf.predict_proba(X_test)[:, 1])
print("=== Random Forest ===")
print(f"Train accuracy: {rf_train_acc:.4f}")
print(f"Test accuracy : {rf_test_acc:.4f}")
print(f"Test AUC      : {rf_auc:.4f}")

# Feature importances
# Need to extract the trained RandomForest from the pipeline
rf_model = rf.named_steps['clf']
# Get feature names after preprocessing
preproc = rf.named_steps['preproc']
# Transform training data to get feature names
X_train_trans = preproc.transform(X_train)
preprocessor_obj = preproc.named_steps['preprocessor']
# Get feature names after preprocessing
try:
    feature_names = preprocessor_obj.get_feature_names_out()
except AttributeError:
    # fallback: use generic names
    feature_names = [f"feat_{i}" for i in range(X_train_trans.shape[1])]

importances = rf_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
print("\nTop 5 feature importances:")
for i in range(5):
    print(f"{feature_names[sorted_idx[i]]}: {importances[sorted_idx[i]]:.4f}")


=== Random Forest ===
Train accuracy: 0.7033
Test accuracy : 0.6995
Test AUC      : 0.7775

Top 5 feature importances:
remainder__Quantity: 0.7754
remainder__Customer ID: 0.2075
Country__Country_United Kingdom: 0.0121
Country__Country_infrequent_sklearn: 0.0028
Country__Country_Germany: 0.0011


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier # Ensure these are imported
from sklearn.tree import DecisionTreeClassifier # Ensure this is imported
from sklearn.metrics import accuracy_score, roc_auc_score # Ensure these are imported
from sklearn.pipeline import Pipeline # Ensure this is imported
from sklearn.impute import SimpleImputer # Ensure this is imported
from sklearn.preprocessing import StandardScaler, OneHotEncoder # Ensure these are imported
from sklearn.compose import ColumnTransformer # Ensure this is imported


# ------------------------------------------------------------
# 7. Gradient Boosting (n_estimators=100, learning_rate=0.1, max_depth=3)
# ------------------------------------------------------------

# --- Start of modifications for NaN handling ---
# Re-define preprocessing pipeline elements locally to include SimpleImputer for numerical columns.
# This addresses the ValueError: Input X contains NaN for GradientBoostingClassifier.
# This duplicates code from previous cells but adheres to the instruction to only modify this cell.

# Re-declaring key variables from earlier cells to ensure local scope for this fix
cat_cols = ['Invoice', 'StockCode', 'Description', 'InvoiceDate', 'Country']
num_cols = ['Quantity', 'Customer ID']
# ordinal_mappings = {} # Based on previous cell, it was empty, no need to redefine if not used

# Define preprocessing pipelines for numerical and categorical features
numerical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), # Impute NaNs in categorical columns
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore',
                             sparse_output=False,
                             min_frequency=0.01, max_categories=20))
])

# Create the ColumnTransformer
preprocessor_fixed = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ],
    remainder='passthrough' # Any columns not specified will be passed through (e.g., if any new ones appear)
)

# Re-create the base_preprocessor. Note: StandardScaler is now part of numerical_transformer,
# so it's not needed at the top level of base_preprocessor anymore.
base_preprocessor_fixed = Pipeline([
    ('preprocessor', preprocessor_fixed)
])

# Re-create the make_full_pipeline function to use the fixed base_preprocessor
def make_full_pipeline_fixed(classifier):
    return Pipeline([
        ('preproc', base_preprocessor_fixed),
        ('clf', classifier)
    ])
# --- End of modifications for NaN handling ---


gb = make_full_pipeline_fixed(GradientBoostingClassifier( # Use the new fixed pipeline function
    n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
))
gb.fit(X_train, y_clf_train)
gb_train_acc = accuracy_score(y_clf_train, gb.predict(X_train))
gb_test_acc = accuracy_score(y_clf_test, gb.predict(X_test))
gb_auc = roc_auc_score(y_clf_test, gb.predict_proba(X_test)[:, 1])
print("\n=== Gradient Boosting ===")
print(f"Train accuracy: {gb_train_acc:.4f}")
print(f"Test accuracy : {gb_test_acc:.4f}")
print(f"Test AUC      : {gb_auc:.4f}")



=== Gradient Boosting ===
Train accuracy: 0.7080
Test accuracy : 0.7057
Test AUC      : 0.7803


In [ ]:
# ------------------------------------------------------------
# 8. Feature ablation (remove 5 lowest‑importance features)
# ------------------------------------------------------------

# Re-create the Random Forest pipeline and calculate its AUC
# This ensures 'rf' and 'rf_auc' are defined even if previous cells are not run in order
rf = make_full_pipeline(RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42
))
rf.fit(X_train, y_clf_train)
rf_auc = roc_auc_score(y_clf_test, rf.predict_proba(X_test)[:, 1])

# Re-extract feature importances and names from the RF model
rf_model = rf.named_steps['clf']
preproc = rf.named_steps['preproc']

X_train_trans = preproc.transform(X_train)
preprocessor_obj = preproc.named_steps['preprocessor']

try:
    feature_names = preprocessor_obj.get_feature_names_out()
except AttributeError:
    feature_names = [f"feat_{i}" for i in range(X_train_trans.shape[1])]

importances = rf_model.feature_importances_

# Get importance and sort ascending
lowest_idx = np.argsort(importances)[:5]   # indices of lowest 5
lowest_feature_names = [feature_names[i] for i in lowest_idx]
print(f"\nRemoving features: {lowest_feature_names}")

# We need to remove these from the preprocessed data (scaled)
# We'll refit the preprocessor on training to get transformed data
X_train_trans = preproc.transform(X_train)
X_test_trans = preproc.transform(X_test)
# Create reduced versions without those columns
X_train_reduced = np.delete(X_train_trans, lowest_idx, axis=1)
X_test_reduced = np.delete(X_test_trans, lowest_idx, axis=1)

# Train a new Random Forest on reduced data (no pipeline for simplicity)
rf_reduced = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42
)
rf_reduced.fit(X_train_reduced, y_clf_train)
rf_reduced_auc = roc_auc_score(y_clf_test, rf_reduced.predict_proba(X_test_reduced)[:, 1])

print(f"AUC full model    : {rf_auc:.4f}")
print(f"AUC reduced model : {rf_reduced_auc:.4f}")



Removing features: ['Country__Country_France', 'Country__Country_Germany', 'Country__Country_infrequent_sklearn', 'Country__Country_United Kingdom', 'remainder__Customer ID']
AUC full model    : 0.7797
AUC reduced model : 0.7265


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score # Added imports

# ------------------------------------------------------------
# 9. Cross‑validated comparison (5‑fold StratifiedKFold)
# ------------------------------------------------------------
# Models to compare: Logistic Regression (from Part 2), controlled DT, RF, GBoost
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': make_full_pipeline_fixed(LogisticRegression(max_iter=1000, random_state=42)),
    'Decision Tree (controlled)': make_full_pipeline_fixed(DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)),
    'Random Forest': make_full_pipeline_fixed(RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)),
    'Gradient Boosting': make_full_pipeline_fixed(GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))
}

print("\n=== Cross‑validated AUC (mean \u00b1 std) ===")
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_clf_train, cv=cv, scoring='roc_auc')
    cv_results[name] = scores
    print(f"{name:25s}: {scores.mean():.4f} \u00b1 {scores.std():.4f}")



=== Cross‑validated AUC (mean ± std) ===
Logistic Regression      : 0.6949 ± 0.0029
Decision Tree (controlled): 0.7536 ± 0.0033
Random Forest            : 0.7861 ± 0.0011
Gradient Boosting        : 0.7810 ± 0.0012


In [ ]:
from sklearn.model_selection import GridSearchCV # Added import

# ------------------------------------------------------------
# 10. GridSearchCV on Random Forest (with full Pipeline on raw data)
# ------------------------------------------------------------

# Re-declaring key variables and pipeline components for robustness within this cell
# This ensures proper NaN handling during GridSearchCV
cat_cols = ['Invoice', 'StockCode', 'Description', 'InvoiceDate', 'Country']
num_cols = ['Quantity', 'Customer ID']

# Define preprocessing pipelines for numerical and categorical features
numerical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore',
                             sparse_output=False,
                             min_frequency=0.01, max_categories=20))
])

# Create the ColumnTransformer
preprocessor_grid_search = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ],
    remainder='passthrough'
)

# Re-create the base_preprocessor to include imputation
base_preprocessor_grid_search = Pipeline([
    ('preprocessor', preprocessor_grid_search)
])

# We'll build a pipeline with the preprocessor + RF
pipe_rf = Pipeline([
    ('preproc', base_preprocessor_grid_search),
    ('rf', RandomForestClassifier(random_state=42))
])

param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [5, 10, None],
    'rf__min_samples_leaf': [1, 5]
}

grid_search = GridSearchCV(
    pipe_rf, param_grid, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_clf_train)

print("\n=== GridSearchCV Results ===")
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.4f}")

best_pipeline = grid_search.best_estimator_


Fitting 5 folds for each of 18 candidates, totalling 90 fits

=== GridSearchCV Results ===
Best params: {'rf__max_depth': None, 'rf__min_samples_leaf': 1, 'rf__n_estimators': 200}
Best CV AUC: 0.8426


In [ ]:
# ------------------------------------------------------------
# 11. Manual learning curve (best pipeline on 20%‑100% of training)
# ------------------------------------------------------------

best_pipeline = None # Initialize to ensure it's always defined
fallback_used = False

if 'grid_search' in globals():
    try:
        # Attempt to get best_estimator_ from grid_search
        best_pipeline = grid_search.best_estimator_
        if best_pipeline is None: # Additional check if best_estimator_ was somehow set to None
            raise AttributeError("grid_search.best_estimator_ is None")
    except AttributeError:
        # If grid_search exists but best_estimator_ is not available (e.g., due to KeyboardInterrupt)
        print("Warning: 'grid_search.best_estimator_' not found. Initializing 'best_pipeline' with a default Random Forest.")
        print("Please ensure Cell 10 (GridSearchCV) completes for the actual best pipeline.")
        fallback_used = True
else:
    # If grid_search object itself is not in globals()
    print("Warning: 'grid_search' not found. Initializing 'best_pipeline' with a default Random Forest.")
    print("Please ensure Cell 10 (GridSearchCV) completes for the actual best pipeline.")
    fallback_used = True

# If fallback is needed, define best_pipeline with a default model
if fallback_used:
    # Re-create make_full_pipeline_fixed and its dependencies if not available
    # This block ensures 'make_full_pipeline_fixed' and its dependencies are present
    # even if previous cells were skipped or modified.
    if 'make_full_pipeline_fixed' not in globals():
        cat_cols = ['Invoice', 'StockCode', 'Description', 'InvoiceDate', 'Country']
        num_cols = ['Quantity', 'Customer ID']

        numerical_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ])

        categorical_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore',
                                     sparse_output=False,
                                     min_frequency=0.01, max_categories=20))
        ])

        preprocessor_fixed = ColumnTransformer(
            transformers=[
                ('num', numerical_transformer, num_cols),
                ('cat', categorical_transformer, cat_cols)
            ],
            remainder='passthrough'
        )

        base_preprocessor_fixed = Pipeline([
            ('preprocessor', preprocessor_fixed)
        ])

        def make_full_pipeline_fixed(classifier):
            return Pipeline([
                ('preproc', base_preprocessor_fixed),
                ('clf', classifier)
            ])

    best_pipeline = make_full_pipeline_fixed(
        RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    )

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
train_aucs = []
test_aucs = []

for f in fractions:
    size = int(f * len(X_train))
    X_train_sub = X_train.iloc[:size]
    y_train_sub = y_clf_train.iloc[:size]
    # Clone the best pipeline and fit on subset
    pipe_clone = Pipeline(best_pipeline.steps)
    pipe_clone.fit(X_train_sub, y_train_sub)
    # Training AUC on the same subset
    train_pred_proba = pipe_clone.predict_proba(X_train_sub)[:, 1]
    train_auc = roc_auc_score(y_train_sub, train_pred_proba)
    # Test AUC on full test set
    test_pred_proba = pipe_clone.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_clf_test, test_pred_proba)
    train_aucs.append(train_auc)
    test_aucs.append(test_auc)

print("\n=== Learning Curve ===")
print("Fraction | Train AUC | Test AUC")
for f, ta, tea in zip(fractions, train_aucs, test_aucs):
    print(f"{f:8.1f} | {ta:.4f}    | {tea:.4f}")



=== Learning Curve ===
Fraction | Train AUC | Test AUC
     0.2 | 0.9078    | 0.8066
     0.4 | 0.8962    | 0.8267
     0.6 | 0.8901    | 0.8368
     0.8 | 0.8865    | 0.8430
     1.0 | 0.8841    | 0.8459


In [11]:

# 12. Serialize best model
joblib.dump(best_pipeline, 'best_model.pkl')
print("\nModel saved as 'best_model.pkl'")




Model saved as 'best_model.pkl'


In [12]:
# 13. Reload and predict on two rows
loaded_model = joblib.load('best_model.pkl')
sample_rows = X.iloc[:2]   # raw rows
preds = loaded_model.predict(sample_rows)
probas = loaded_model.predict_proba(sample_rows)[:, 1]
print("Predictions on first two rows:")
print(f"Classes: {preds}")
print(f"Probabilities: {probas}")

Predictions on first two rows:
Classes: [0 0]
Probabilities: [0.15415826 0.15415826]


In [13]:
import joblib
model = joblib.load('best_model.pkl')
sample = X.iloc[:2]
print(model.predict(sample))

[0 0]
